# Regresión y correlación múltiple para clínicos
### Ejemplo con datos abiertos de la NASA: tripulación civil de Inspiration4

**Autor:** Víctor Junior Godínez-Vázquez · Material docente · Licencia CC BY 4.0 (texto) / MIT (código)

**Cómo usar este cuaderno:** menú **Entorno de ejecución → Ejecutar todas** (o `Ctrl + F9`). No necesita instalar nada.

**Pregunta clínica:** ¿cambian la glucosa, el potasio, la creatinina y las proteínas totales con un vuelo orbital de 3 días, comparando a cada tripulante consigo mismo?

**Qué aprenderá**
1. Una variable categórica (momento del viaje) se convierte en *variables indicadoras* (dummies).
2. Qué cambia cuando cada persona se mide varias veces (**medidas repetidas**).
3. Comparar tres modelos: **A** ingenuo, **B** regresión múltiple con el tripulante, **C** modelo mixto.
4. Revisar la calidad de los datos **antes** de analizar, y los supuestos con los **residuos**.

> **Advertencia:** análisis **exploratorio y didáctico**. Estos cuatro análisis de laboratorio no formaron parte del análisis preregistrado del estudio original (doi:10.17605/OSF.IO/W3NP2). Con 4 personas, ningún resultado debe usarse para decisiones clínicas.

**Datos:** NASA Open Science Data Repository, estudio OSD-575 (doi:10.26030/mc5d-p710). Archivo derivado `i4_analysis.csv`, disponible en https://osf.io/57cdk

## 1. Preparar el entorno
Cargamos las librerías. `statsmodels` es la librería de Python para modelos estadísticos (equivale a los menús de SPSS).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
print("Listo ✔")

## 2. Cargar los datos
El cuaderno intenta descargar el archivo del repositorio de GitHub. Si no lo logra, le pedirá subir `i4_analysis.csv` desde su computadora.

In [ ]:
BASE = "https://raw.githubusercontent.com/ViiJX-stack/regression-multiple-clinicos/main/"
d = None
for ruta in ["i4_analysis.csv", "datos/i4_analysis.csv"]:
    try:
        d = pd.read_csv(BASE + ruta)
        print("Datos descargados de GitHub ✔")
        break
    except Exception:
        pass
if d is None:
    from google.colab import files
    print("No se pudo descargar. Suba el archivo i4_analysis.csv:")
    subido = files.upload()
    d = pd.read_csv(list(subido.keys())[0])

# Nombres en español para el momento del viaje (el prefijo numérico fija el orden; 'antes' es la referencia)
d["momento"] = d["phase"].map({"1_pre": "1_antes", "2_return": "2_regreso", "3_recovery": "3_recuperacion"})
d["tripulante"] = d["crew"]
ORDEN = ["L-92", "L-44", "L-3", "R+1", "R+45", "R+82", "R+194"]
print(d.shape[0], "filas =", d.tripulante.nunique(), "tripulantes ×", d.timepoint.nunique(), "momentos")
d[["tripulante", "timepoint", "momento", "glucose_raw", "potassium_raw", "creatinine", "total_protein"]]

**Lectura de la tabla.** Cada fila es una muestra de sangre. Hay **28 filas pero solo 4 personas**: cada tripulante aparece 7 veces.
- `L-92` = 92 días antes del lanzamiento · `R+1` = 1 día después del regreso.
- Unidades: glucosa mg/dL · potasio mmol/L · creatinina mg/dL · proteínas totales g/dL.

## 3. Calidad de los datos: primero revisar, después analizar
Observe la glucosa en **R+45**. Valores < 55 mg/dL en adultos asintomáticos no son fisiológicos: indican que la muestra se procesó tarde y las células **consumieron la glucosa en el tubo** (glucólisis in vitro). El mismo retraso puede **elevar falsamente el potasio** (salida de potasio de las células).

In [ ]:
r45 = d[d.timepoint == "R+45"][["tripulante", "glucose_raw", "potassium_raw"]]
print("Glucosa y potasio en R+45 (valores crudos):")
print(r45.to_string(index=False))
print("\nRegla fijada antes del análisis: glucosa < 55 mg/dL → excluir glucosa y potasio de esa toma.")
# Las columnas *_analysis ya aplican la regla (R+45 = vacío)
LABS = {"glucose_analysis": "Glucosa (mg/dL)", "potassium_analysis": "Potasio (mmol/L)",
        "creatinine": "Creatinina (mg/dL)", "total_protein": "Proteínas totales (g/dL)"}
print("\nValores disponibles por análisis:", {v: int(d[k].notna().sum()) for k, v in LABS.items()})

## 4. Mirar los datos: una línea por tripulante
Antes de cualquier modelo, **grafique**. Cada color es una persona; la banda gris marca el día siguiente al regreso.

In [ ]:
colores = {"C001": "#2a78d6", "C002": "#eb6834", "C003": "#1baf7a", "C004": "#c98500"}
fig, ejes = plt.subplots(2, 2, figsize=(11, 7))
x = np.arange(len(ORDEN))
for ax, (col, nombre) in zip(ejes.ravel(), LABS.items()):
    ax.axvspan(2.5, 3.5, color="#eeeeee")
    for t, g in d.groupby("tripulante"):
        g = g.set_index("timepoint").loc[ORDEN]
        ax.plot(x, g[col], marker="o", color=colores[t], label=t)
    ax.set_xticks(x); ax.set_xticklabels(ORDEN, fontsize=8); ax.set_title(nombre)
ejes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Pregunta para el grupo:** ¿las líneas se separan más **entre personas** o **entre momentos**? Por ejemplo, en la creatinina C002 está siempre más bajo que C001. Esa diferencia entre personas es la clave de este ejercicio.

## 5. Variables categóricas → variables indicadoras (dummies)
La regresión necesita números. El *momento del viaje* tiene 3 categorías, así que se convierte en **2 columnas 0/1**; la tercera categoría (**antes**) es la **referencia**. SPSS hace lo mismo internamente.

In [ ]:
dummies = pd.get_dummies(d["momento"], drop_first=True).astype(int)
pd.concat([d[["tripulante", "timepoint", "momento"]], dummies], axis=1).head(7)

**Interpretación:**
`y = β0 + β1·regreso + β2·recuperación`
- **β0** = promedio **antes** del vuelo.
- **β1** = diferencia **regreso − antes**.
- **β2** = diferencia **recuperación − antes**.

## 6. Tres modelos para la misma pregunta
| Modelo | Fórmula | Qué supone |
|---|---|---|
| **A. Ingenuo** | `y ~ momento` | 28 observaciones independientes (falso: son 4 personas) |
| **B. Regresión múltiple** | `y ~ momento + tripulante` | Cada persona tiene su propio nivel basal → compara a cada uno consigo mismo |
| **C. Modelo mixto** | `y ~ momento`, intercepto aleatorio por tripulante | Igual que B, pero trata a los tripulantes como muestra de una población |

El modelo **B** es una regresión múltiple: 2 dummies de momento + 3 dummies de tripulante = **5 variables independientes**.

In [ ]:
def ajustar(col):
    datos = d[["tripulante", "momento", col]].dropna().rename(columns={col: "y"})
    mA = smf.ols("y ~ C(momento)", data=datos).fit()
    mB = smf.ols("y ~ C(momento) + C(tripulante)", data=datos).fit()
    mC = smf.mixedlm("y ~ C(momento)", data=datos, groups=datos["tripulante"]).fit(reml=True)
    return datos, mA, mB, mC

def fila(modelo, nombre, termino):
    return {"modelo": nombre, "coef": modelo.params[termino], "EE": modelo.bse[termino], "p": modelo.pvalues[termino]}

resultados = []
modelos = {}
for col, nombre in LABS.items():
    datos, mA, mB, mC = ajustar(col)
    modelos[col] = (datos, mA, mB, mC)
    var_trip = float(mC.cov_re.iloc[0, 0]); var_res = float(mC.scale)
    icc = var_trip / (var_trip + var_res)
    for termino, etiqueta in [("C(momento)[T.2_regreso]", "regreso − antes"), ("C(momento)[T.3_recuperacion]", "recuperación − antes")]:
        for m, nm in [(mA, "A ingenuo"), (mB, "B + tripulante"), (mC, "C mixto")]:
            r = fila(m, nm, termino); r.update({"análisis": nombre, "contraste": etiqueta, "n": len(datos)})
            r["R² (A/B)"] = getattr(m, "rsquared", np.nan); r["CCI"] = icc if nm == "C mixto" else np.nan
            resultados.append(r)

tabla = pd.DataFrame(resultados)[["análisis", "contraste", "modelo", "n", "coef", "EE", "p", "R² (A/B)", "CCI"]]
tabla.round(3)

### Cómo leer la tabla
- **coef**: diferencia promedio en las unidades del análisis (por ejemplo, mg/dL).
- **EE** (error estándar): incertidumbre del coeficiente. **Compare el EE de A contra B y C.**
- **R²**: proporción de la variabilidad explicada. Su raíz cuadrada es el **coeficiente de correlación múltiple (R)**.
- **CCI** (coeficiente de correlación intraclase): qué parte de la variabilidad se debe a **diferencias entre personas**. Si es alto, ignorar a la persona (modelo A) es un error grave.

In [ ]:
print("Resumen por análisis (contraste regreso − antes):\n")
for col, nombre in LABS.items():
    t = tabla[(tabla["análisis"] == nombre) & (tabla.contraste == "regreso − antes")].set_index("modelo")
    cci = t.loc["C mixto", "CCI"]
    print(f"{nombre}: CCI = {cci:.2f} → {cci*100:.0f}% de la variabilidad se debe a diferencias entre personas")
    print(f"   EE modelo A = {t.loc['A ingenuo','EE']:.3f} | EE modelo B = {t.loc['B + tripulante','EE']:.3f}"
          f" | p A = {t.loc['A ingenuo','p']:.3f} | p B = {t.loc['B + tripulante','p']:.3f} | p C = {t.loc['C mixto','p']:.3f}")
    print(f"   R² A = {t.loc['A ingenuo','R² (A/B)']:.2f} → R múltiple = {np.sqrt(t.loc['A ingenuo','R² (A/B)']):.2f} | "
          f"R² B = {t.loc['B + tripulante','R² (A/B)']:.2f} → R múltiple = {np.sqrt(t.loc['B + tripulante','R² (A/B)']):.2f}\n")

**Mensaje central:** cuando las personas difieren mucho entre sí (CCI alto), el modelo **A** mezcla esa variabilidad con el error, su **EE crece** y puede **ocultar** un efecto real. Los modelos **B** y **C** comparan a cada persona consigo misma.

**Lo que muestran estos datos**
- **Creatinina y potasio:** CCI de 0.67 y 0.59. Al incluir al tripulante (modelo B), el EE baja entre 33% y 40% y el R² sube de 0.01 a 0.68 en la creatinina. Ese es el efecto de comparar a cada persona consigo misma.
- **Glucosa:** subió unos 18 mg/dL el día siguiente al regreso, **en los 4 tripulantes**. Aquí el CCI es ≈ 0, así que los tres modelos coinciden. Es coherente con una respuesta de estrés (hormonas contrarreguladoras), pero **no sabemos si las muestras eran en ayuno**, y algunas basales también son bajas (56 y 62 mg/dL), posiblemente por el mismo problema del tubo. Es un hallazgo **exploratorio**.
- **Comparaciones múltiples:** son 4 análisis × 2 contrastes = 8 pruebas. Con la corrección de Bonferroni el umbral sería 0.05 / 8 ≈ 0.006.
- **Proteínas totales:** sin cambios y sin diferencias entre personas (CCI ≈ 0).

**Nota técnica:** el modelo mixto de Python (`statsmodels`) usa una aproximación normal para los valores *p*. Con tan pocos grupos, el artículo original usó la corrección de Kenward-Roger en R, que es más conservadora. Para fines de enseñanza la diferencia no cambia el mensaje.

## 7. Supuestos: se revisan con los **residuos**, no con las variables
El supuesto de normalidad se refiere a los **errores del modelo** (residuos), no a que cada variable tenga distribución normal. Revisamos el modelo B de la creatinina.

In [ ]:
datos, mA, mB, mC = modelos["creatinine"]
res, ajust = mB.resid, mB.fittedvalues
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
a1.scatter(ajust, res); a1.axhline(0, color="gray", ls="--")
a1.set_xlabel("Valores ajustados"); a1.set_ylabel("Residuos"); a1.set_title("Residuos vs ajustados (varianza constante)")
sm.qqplot(res, line="s", ax=a2); a2.set_title("Gráfico Q-Q (normalidad de residuos)")
plt.tight_layout(); plt.show()

**Qué buscar:** a la izquierda, una nube sin forma de embudo; a la derecha, puntos cerca de la línea. Con 28 datos, pequeñas desviaciones son esperables; las pruebas formales de normalidad tienen poco poder con muestras tan pequeñas.

## 8. ¿Y la correlación?
La correlación de Pearson mide la asociación lineal entre dos variables. **Cuidado:** si calcula la correlación con las 28 filas, mezcla diferencias entre personas con cambios dentro de cada persona.

In [ ]:
cols = ["glucose_analysis", "potassium_analysis", "creatinine", "total_protein"]
print("Correlación de Pearson con las 28 filas (mezcla entre y dentro de personas):")
print(d[cols].corr().round(2))
centrado = d[cols].sub(d.groupby("tripulante")[cols].transform("mean"))
print("\nCorrelación DENTRO de cada persona (restando el promedio de cada tripulante):")
print(centrado.corr().round(2))

**Ejemplo en estos datos:** creatinina y potasio tienen r = 0.45 con las 28 filas, pero r ≈ 0 dentro de cada persona. La "asociación" venía de que los tripulantes con creatinina más alta también tienden a tener potasio más alto, no de que ambos cambien juntos en la misma persona. Es el mismo principio de los modelos A y B: **la persona es un factor de confusión**.

## 9. Lo mismo en SPSS
- **Modelo B (momento + tripulante):** Analizar → Modelo lineal general → Univariado. Variable dependiente: el análisis; Factores fijos: momento y tripulante.
- **Modelo C (mixto):** Analizar → Modelos mixtos → Lineal. Sujetos: tripulante; Repetidas: momento de toma. Variable dependiente: el análisis; Factores: momento; Efectos aleatorios: intersección por tripulante.

**Mensaje:** el programa no importa; importa construir el modelo correcto.

## 10. Preguntas frecuentes
**¿Por qué no un ANOVA de una vía?** Es el modelo A: supone 28 observaciones independientes. El ANOVA de medidas repetidas sí sería válido, pero exige datos completos (sin valores excluidos, como R+45) y es menos flexible que el modelo mixto.

**¿Por qué no una t de Student pareada?** Solo compara dos momentos y desperdicia las tres tomas basales. El modelo usa todas.

**¿n = 4 no es muy poco?** Sí para generalizar, pero es **toda la tripulación** de la primera misión orbital completamente civil. El análisis correcto con pocas personas es exactamente el que compara a cada una consigo misma.

**¿Esto prueba que el vuelo cambió los análisis?** No. No hay grupo control en tierra, y estos análisis son exploratorios.

## Referencias
1. Overbey EG, et al. The Space Omics and Medical Atlas (SOMA) and international astronaut biobank. *Nature*. 2024;632:1145-54.
2. Jones CW, et al. Molecular and physiological changes in the SpaceX Inspiration4 civilian crew. *Nature*. 2024;632:1155-64.
3. Godínez-Vázquez VJ. Change in the neutrophil-to-lymphocyte ratio after a 3-day orbital flight [preprint]. *medRxiv*. 2026. doi:10.64898/2026.10.16.26365757
4. UNC Eshelman School of Pharmacy. Biostatistics, Lesson 6: Continuous outcomes, linear regression (lectura previa).

*Este cuaderno se elaboró con asistencia de IA (Claude, Anthropic); el autor revisó el contenido y es responsable de él.*